In [ ]:
from __future__ import annotations
import os
import sys
import json
import time
from pathlib import Path
from typing import Dict, List, Tuple, Set

import pandas as pd

# --- caminhos padrão (ajuste se necessário) ---
DATA_DIR = Path("data")
DATE = "20251112"
ASSIGNMENTS_FILE = DATA_DIR / "assignments.csv"
USERS_FILE = DATA_DIR / "users_hashed.csv"
ANNOTATIONS_DIR = DATA_DIR / f"annotations/{DATE}"
REPORTS_DIR = DATA_DIR / "reports"

# prefixo dos arquivos de anotação
ANNOTATION_PREFIX = "annotation_"  # arquivo: annotation_<username>.csv

# máximo de usuários no relatório (se USERS_FILE tiver mais)
MAX_USERS = 35

# rótulos esperados (ajuste se necessário)
VALID_LABELS = {"sim", "nao", "nao_sei"}


def read_csv_loose(path: Path, usecols: List[str] | None = None) -> pd.DataFrame:
    """
    Lê CSV com dtype=str, engine=python e tolerância a separador/encoding comum.
    """
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    last_err = None
    for enc in ("utf-8", "utf-8-sig", "latin1"):
        try:
            df = pd.read_csv(path, dtype=str, engine="python", sep=None, encoding=enc)
            if usecols:
                # seleção independente de ordem / nomes normalizados
                norm = {str(c).strip().lower(): c for c in df.columns}
                missing = [c for c in usecols if c not in norm]
                if missing:
                    raise ValueError(
                        f"{path.name}: faltam colunas {missing}. Encontradas: {list(df.columns)}"
                    )
                df = df[[norm[c] for c in usecols]].copy()
                df.columns = usecols  # normaliza nomes
            return df
        except Exception as e:
            last_err = e
    raise ValueError(f"Falha ao ler {path} com encodings comuns. Último erro: {last_err}")


def load_assignments() -> pd.DataFrame:
    df = read_csv_loose(ASSIGNMENTS_FILE)
    # normaliza nomes
    df.columns = [str(c).strip().lower() for c in df.columns]
    # exige colunas
    required = ["annotator_id", "frase_id"]
    norm = {c: c for c in df.columns}
    for req in required:
        if req not in norm:
            raise ValueError(f"{ASSIGNMENTS_FILE.name}: precisa de colunas {required}. Tem: {list(df.columns)}")
    df = df[required].copy()
    # tipagem leve
    return df.astype({"annotator_id": "string", "frase_id": "string"})


def load_users_limit(assignments_df: pd.DataFrame) -> List[str]:
    """
    Define a lista de usuários a reportar:
    1) Se users_hashed.csv existir, usa a coluna 'username' (até 35).
    2) Senão, usa os 'annotator_id' únicos do assignments.
    3) Se não houver nada, infere pelos arquivos em data/annotations/.
    """
    if USERS_FILE.exists():
        df = read_csv_loose(USERS_FILE)
        df.columns = [str(c).strip().lower() for c in df.columns]
        if "username" not in df.columns:
            raise ValueError(f"{USERS_FILE.name} precisa conter a coluna 'username'. Colunas: {list(df.columns)}")
        users = df["username"].astype("string").dropna().str.strip()
        users = [u for u in users if u]
        if len(users) > MAX_USERS:
            users = users[:MAX_USERS]
        return users

    # fallback: assignments
    ann_users = assignments_df["annotator_id"].dropna().astype(str).str.strip().unique().tolist()
    if ann_users:
        return ann_users

    # último fallback: pelos arquivos no dir de annotations
    if ANNOTATIONS_DIR.exists():
        users = []
        for p in ANNOTATIONS_DIR.glob(f"{ANNOTATION_PREFIX}*.csv"):
            name = p.stem.replace(ANNOTATION_PREFIX, "")
            if name:
                users.append(name)
        return sorted(set(users))

    return []


def annotation_file_for_user(user: str) -> Path:
    safe_user = "".join(ch for ch in str(user) if ch.isalnum() or ch in ("_", "-", "."))
    return ANNOTATIONS_DIR / f"{ANNOTATION_PREFIX}{safe_user}.csv"


def load_user_annotations(user: str) -> Tuple[pd.DataFrame, float | None]:
    """
    Carrega anotações do usuário (se existir) e retorna (df, mtime).
    df terá colunas: user, frase_id, annotation (string).
    mtime é o timestamp do arquivo em segundos, ou None.
    """
    f = annotation_file_for_user(user)
    if not f.exists():
        df = pd.DataFrame(
            {"user": pd.Series(dtype="string"),
             "frase_id": pd.Series(dtype="string"),
             "annotation": pd.Series(dtype="string")}
        )
        return df, None
    df = read_csv_loose(f)
    df.columns = [str(c).strip().lower() for c in df.columns]
    required = ["user", "frase_id", "annotation"]
    for req in required:
        if req not in df.columns:
            raise ValueError(f"{f.name} deve conter colunas {required}. Colunas: {list(df.columns)}")
    df = df[required].astype({"user": "string", "frase_id": "string", "annotation": "string"})
    # filtra só o próprio user (caso tenha ruído)
    df = df[df["user"] == str(user)]
    try:
        mtime = f.stat().st_mtime
    except Exception:
        mtime = None
    return df, mtime


def summarize_user(user: str, assignments_df: pd.DataFrame) -> Dict:
    """
    Gera métricas de completude para um usuário.
    """
    # frases atribuídas a esse usuário (normalizando para string)
    assigned = (
        assignments_df.loc[assignments_df["annotator_id"] == str(user), "frase_id"]
        .dropna().astype(str).str.strip().unique().tolist()
    )
    assigned_set: Set[str] = set(assigned)

    # anotações do usuário
    ann_df, mtime = load_user_annotations(user)

    # normaliza e deduplica
    ann_df = ann_df.dropna(subset=["frase_id"])
    ann_df["frase_id"] = ann_df["frase_id"].astype(str).str.strip()
    ann_df = ann_df.drop_duplicates(subset=["frase_id"], keep="last")

    # só conta como “anotado” o que está dentro do conjunto atribuído
    annotated_in_assigned = ann_df[ann_df["frase_id"].isin(assigned_set)]
    annotated_set: Set[str] = set(annotated_in_assigned["frase_id"].tolist())

    # pendentes = atribuídas - anotadas
    pending_set = assigned_set - annotated_set

    # extras (anotadas que não foram atribuídas) — útil para auditoria
    extras_set = set(ann_df["frase_id"].tolist()) - assigned_set

    # breakdown de rótulos dentro do conjunto válido
    labels_counts = (
        annotated_in_assigned["annotation"].str.strip().str.lower()
        .value_counts(dropna=False)
        .to_dict()
    )
    # garante chaves para rótulos esperados
    labels_counts = {k: int(labels_counts.get(k, 0)) for k in sorted(VALID_LABELS)}

    pct = (len(annotated_set) / max(1, len(assigned_set))) * 100.0

    return {
        "user": user,
        "assigned_total": int(len(assigned_set)),
        "annotated_total": int(len(annotated_set)),
        "pending": int(len(pending_set)),
        "percent_complete": round(pct, 2),
        "label_sim": labels_counts.get("sim", 0),
        "label_nao": labels_counts.get("nao", 0),
        "label_nao_sei": labels_counts.get("nao_sei", 0),
        "extra_annotations": int(len(extras_set)),
        "last_update": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(mtime)) if mtime else "",
        # campos para arquivo detalhado
        "_pending_ids": sorted(pending_set),
        "_extra_ids": sorted(extras_set),
    }


def main():
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Lendo assignments de {ASSIGNMENTS_FILE} ...")
    assignments_df = load_assignments()

    print("Definindo lista de usuários ...")
    users = load_users_limit(assignments_df)
    if not users:
        print("⚠️  Não encontrei usuários em users_hashed.csv, assignments.csv ou arquivos de anotação.")
        sys.exit(1)

    # limita a 35 se vierem mais
    if len(users) > MAX_USERS:
        users = users[:MAX_USERS]

    print(f"Usuários incluídos no relatório ({len(users)}): {users}")

    rows = []
    pendentes_rows = []  # saída detalhada

    for user in users:
        try:
            summary = summarize_user(user, assignments_df)
            rows.append({k: v for k, v in summary.items() if not k.startswith("_")})

            # detalhe de pendentes
            for fid in summary["_pending_ids"]:
                pendentes_rows.append({"user": user, "frase_id_pendente": fid})
        except Exception as e:
            print(f"Erro processando {user}: {e}")
            rows.append({
                "user": user,
                "assigned_total": 0,
                "annotated_total": 0,
                "pending": 0,
                "percent_complete": 0.0,
                "label_sim": 0, "label_nao": 0, "label_nao_sei": 0,
                "extra_annotations": 0,
                "last_update": "",
            })

    # DataFrame final
    report_df = pd.DataFrame(rows)
    # ordem de colunas amigável
    cols = [
        "user", "assigned_total", "annotated_total", "pending", "percent_complete",
        "label_sim", "label_nao", "label_nao_sei", "extra_annotations", "last_update",
    ]
    report_df = report_df.reindex(columns=cols)

    # ordena por % completo desc e depois por user
    report_df = report_df.sort_values(by=["percent_complete", "user"], ascending=[False, True])

    out_csv = REPORTS_DIR / "completude.csv"
    report_df.to_csv(out_csv, index=False)
    print(f"✅ Relatório salvo em: {out_csv}")

    # arquivo detalhado com pendentes
    pendentes_df = pd.DataFrame(pendentes_rows)
    out_pend = REPORTS_DIR / "completude_detalhe_pendentes.csv"
    pendentes_df.to_csv(out_pend, index=False)
    print(f"🧩 Detalhe de pendentes salvo em: {out_pend}")

    # mostra um resumo no console
    total_assigned = int(report_df["assigned_total"].sum())
    total_annotated = int(report_df["annotated_total"].sum())
    pct_global = (total_annotated / max(1, total_assigned)) * 100.0
    print(f"\nResumo global: {total_annotated}/{total_assigned} ({pct_global:.2f}%) concluído.")

In [10]:
main()

Lendo assignments de data/assignments.csv ...
Definindo lista de usuários ...
Usuários incluídos no relatório (35): ['user01', 'user02', 'user03', 'user04', 'user05', 'user06', 'user07', 'user08', 'user09', 'user10', 'user11', 'user12', 'user13', 'user14', 'user15', 'user16', 'user17', 'user18', 'user19', 'user20', 'user21', 'user22', 'user23', 'user24', 'user25', 'user26', 'user27', 'user28', 'user29', 'user30', 'user31', 'user32', 'user33', 'user34', 'user35']
✅ Relatório salvo em: data/reports/completude.csv
🧩 Detalhe de pendentes salvo em: data/reports/completude_detalhe_pendentes.csv

Resumo global: 46233/70000 (66.05%) concluído.
